# MySQL Contact → OneBill Migration

This notebook migrates **contact records** from MySQL into OneBill by PUTting each contact directly to its subscriber account.

The `AccountCode` in MySQL maps 1-to-1 with the `accountNumber` in OneBill, so no pre-lookup step is needed — each contact is sent straight to `/rest/SubscriberService/v1/subscribers/{AccountCode}`.

**Pipeline:**
1. Load Consumer contacts from MySQL
2. Obtain and cache an OAuth bearer token
3. Build a contact JSON payload per row
4. PUT each contact to OneBill concurrently via a thread pool
5. Log every outcome and export failures to CSV

> **Before running in production:** remove the `df.head(10)` limit in the *Load Data* cell.

## Imports

In [1]:
# %pip install mysql-connector-python sqlalchemy python-dotenv
import json
import os
import threading
import time
import logging
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(override=True)  # .env values take precedence over existing environment variables

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 11


True

## Configuration

All secrets are read from environment variables (`.env` file). Set the following before running:

| Variable | Description |
|---|---|
| `DB_USERNAME` | MySQL username |
| `DB_PASSWORD` | MySQL password |
| `DB_HOST` | MySQL host |
| `CLIENT_ID` | OneBill OAuth client ID |
| `CLIENT_SECRET` | OneBill OAuth client secret |
| `API_USERNAME` | OneBill API username |
| `API_PASSWORD` | OneBill API password |
| `PROXY_ACCOUNT_NUMBER` | OneBill partner/proxy account number |

In [2]:
MAX_WORKERS = 20
TOKEN_TTL_SECONDS = 3500  # Refresh token 100s before expiry (OAuth TTL is typically 3600s)

db_url = f'mysql+mysqlconnector://{os.environ["DB_USERNAME"]}:{os.environ["DB_PASSWORD"]}@{os.environ["DB_HOST"]}/bi_curated_views'

BASE_URL = 'https://sandbox-sg.onebillsoftware.com'
ACCESS_TOKEN_URL = f'{BASE_URL}/oauth/token'

## Step 1 — Load Contact Data from MySQL

Queries `dynamics_contact` joined to `reporting_account`, filtered to **Consumer** segment accounts with a non-null `FirstName`.

In [3]:
query = """
SELECT
    contact.*
FROM
    bi_curated_views.dynamics_contact contact
LEFT JOIN
    bi_curated_views.reporting_account account
ON
    contact.`AccountCode` = account.`AccountCode`
WHERE
    account.`AccountSegment` = 'Consumer'
AND
    contact.`FirstName` IS NOT NULL
ORDER BY
    contact.`AccountCode`;
"""

engine = create_engine(db_url)
df = pd.read_sql(query, con=engine)
df = df.head(10)  # <-- REMOVE in production

print(f'Loaded {len(df):,} contact rows from MySQL')
print(f'Unique AccountCodes: {df["AccountCode"].nunique():,}')

Loaded 10 contact rows from MySQL
Unique AccountCodes: 10


## Step 2 — Logging

A timestamped log file is created for each run, alongside console output, giving a full audit trail of every contact migration attempt.

In [4]:
log_filename = f'contact_migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

## Step 3 — Thread-Safe OAuth Token Manager

Caches the bearer token in memory and proactively refreshes it 100 seconds before expiry. A `threading.Lock` ensures only one thread fetches a new token at a time — all others wait briefly then reuse the cached value.

In [5]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:           # only one thread refreshes at a time
            if datetime.now() >= self._expires_at:
                self._refresh()    # others wait at the lock, then see a valid token
            return self._token

    def _refresh(self):
        logger.info('Refreshing OAuth token...')
        response = requests.post(
            ACCESS_TOKEN_URL,
            data={
                'grant_type':    'password',
                'client_id':     os.environ['CLIENT_ID'],
                'client_secret': os.environ['CLIENT_SECRET'],
                'username':      os.environ['API_USERNAME'],
                'password':      os.environ['API_PASSWORD'],
            },
            headers={'Content-Type': 'application/x-www-form-urlencoded'}
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload['access_token']
        ttl = payload.get('expires_in', TOKEN_TTL_SECONDS)
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)  # 100s safety buffer
        logger.info('Token refreshed; valid until %s', self._expires_at.strftime('%H:%M:%S'))


token_manager = TokenManager()

## Step 4 — Contact Payload Builder

Maps a single DataFrame row to the JSON body expected by the OneBill Subscriber Contact API. Handles `None`/`NaN` values so the API never receives a raw `NaT`.

In [6]:
def build_contact_payload(row: pd.Series) -> str:
    """Map a DataFrame row to the OneBill contact JSON payload."""
    # Replace NaN/NaT with None so json.dumps doesn't produce 'NaN' strings
    row = row.where(pd.notna(row), None).to_dict()

    return json.dumps({
        'contact': [
            {
                'id': row['ContactCode'],
                'firstName': row['FirstName'],
                'lastName':  row['LastName'],
                'ContactType': '1001',
                'primaryContact': 'false',
                'billingContact': 'false',
                'communicationPoint': [
                    {
                        'type': 'Email', 
                        'value': row['EmailAddresses']
                    },
                    {
                        'type': 'Phone', 
                        'value': row['PhoneMobile']
                    },
                    {
                        'type': 'CPhone', 
                        'value': row['PhoneHome']
                    }
                ]   
            }
        ]
    })

## Step 5 — OneBill PUT Request

Sends a single `PUT` to `/rest/SubscriberService/v1/subscribers/{account_code}` to add a contact to the matching OneBill subscriber account.

Both `Authorization` and `proxy_accountNumber` are set explicitly on every request — `proxy_accountNumber` scopes the request to accounts under the partner account in OneBill.

In [7]:
def put_contact(session: requests.Session, account_code: str, payload: str) -> dict:
    """PUT a contact to the OneBill subscriber matching account_code.

    Args:
        session:      Shared connection-pooled requests.Session.
        account_code: MySQL AccountCode, which equals the OneBill accountNumber.
        payload:      JSON string from build_contact_payload().

    Returns:
        Parsed JSON response from OneBill.
    """
    # AccountCode in MySQL == accountNumber in OneBill — no ID lookup required
    url = f'{BASE_URL}/rest/SubscriberService/v1/subscribers/{account_code}'

    # Both headers must be present on every request:
    #   Authorization        — authenticates the API call
    #   proxy_accountNumber  — scopes the request to the correct partner account
    headers = {
        'Authorization':      f'Bearer {token_manager.get_token()}',
        'proxy_accountNumber': os.environ['PROXY_ACCOUNT_NUMBER'],
        'Content-Type':       'application/json',
    }

    response = session.put(url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()  # raises HTTPError for 4xx / 5xx
    data = response.json()

    # OneBill can return HTTP 200 with a validation failure body — check explicitly
    validation = data.get('validationResponse', {})
    if not validation.get('successful', True):
        errors = validation.get('validationErrorInfo', [])
        raise ValueError('; '.join(e.get('message', '') for e in errors))

    return data

## Step 6 — Per-Row Worker

Called once per contact row inside the thread pool. Builds the payload, calls the API, and returns a result dict. Timing is split into *build* (CPU) vs *network* (I/O) to make it easy to spot where time is actually being spent.

In [8]:
def migrate_row(row: pd.Series, session: requests.Session) -> dict:
    """Migrate a single contact row to OneBill."""
    account_code = row['AccountCode']
    contact_code = row['ContactCode']
    first_name   = row['FirstName']
    last_name    = row['LastName']

    # Time payload construction separately from the network round-trip
    t0      = time.perf_counter()
    payload = build_contact_payload(row)
    t_build = time.perf_counter() - t0

    try:
        t1       = time.perf_counter()
        put_contact(session, account_code, payload)
        t_net    = time.perf_counter() - t1

        logger.info(f'  [OK] {account_code} — build={t_build*1000:.0f}ms  net={t_net*1000:.0f}ms')
        return {
            'account_code':     account_code,
            'contact_code':     contact_code,
            'first_name':       first_name,
            'last_name':        last_name,
            'status':           'success',
            'error':            None,
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }

    except Exception as e:
        t_net = time.perf_counter() - t1 if 't1' in dir() else 0
        logger.error(f'  [FAIL] {account_code} — {e}')
        return {
            'account_code':     account_code,
            'contact_code':     contact_code,
            'first_name':       first_name,
            'last_name':        last_name,
            'status':           'failed',
            'error':            str(e),
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }

## Step 7 — Migration Orchestrator

Fans all contact rows out across a `ThreadPoolExecutor`, collects results, and logs progress every 50 rows. A profiling summary is printed on completion showing throughput and network latency percentiles.

In [9]:
def migrate(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    """Migrate all contact rows in df to OneBill concurrently."""

    # Shared connection-pooled session — one pool, reused across all threads
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers  # enough connections for all concurrent threads
    )
    session.mount('https://', adapter)

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f'Starting migration of {total:,} contacts with {max_workers} workers...')
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit one future per contact row — session is shared safely across threads
        futures = {
            executor.submit(migrate_row, row, session): row['AccountCode']
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())

            # Progress log every 50 rows and on the final row
            if i % 50 == 0 or i == total:
                ok   = sum(1 for r in results if r['status'] == 'success')
                fail = sum(1 for r in results if r['status'] == 'failed')
                logger.info(f'Progress: {i}/{total} — {ok} ok, {fail} failed')

    wall_elapsed = time.perf_counter() - wall_start
    results_df   = pd.DataFrame(results)
    success      = (results_df['status'] == 'success').sum()
    failed       = (results_df['status'] == 'failed').sum()

    logger.info(
        f'Done in {wall_elapsed:.1f}s — {success} succeeded, {failed} failed. '
        f'(log: {log_filename})'
    )

    # Profiling summary — if net time dominates, the server is the bottleneck
    net_times = results_df.loc[results_df['elapsed_net_ms'] > 0, 'elapsed_net_ms']
    print('\n=== Profiling Summary ===')
    print(f'Total wall time:          {wall_elapsed:.1f}s')
    print(f'Throughput:               {total / wall_elapsed:.1f} contacts/s')
    print(f'Avg build time per row:   {results_df["elapsed_build_ms"].mean():.1f}ms')
    if not net_times.empty:
        print(f'Avg network time per row: {net_times.mean():.1f}ms')
        print(f'Max network time:         {net_times.max():.1f}ms')
        print(f'P95 network time:         {net_times.quantile(0.95):.1f}ms')
    print('=========================')

    return results_df

## Step 8 — Run the Migration

Runs the full pipeline and displays any failed rows inline. All failures are also exported to `Failed_Contact_Migrations.csv` for review and retry.

In [10]:
results_df = migrate(df)

failures = results_df[results_df['status'] == 'failed']
print(f'\nFailed rows ({len(failures)}):')
display(failures)

2026-05-11 11:08:53,908 [INFO] Starting migration of 10 contacts with 20 workers...
2026-05-11 11:08:53,921 [INFO] Refreshing OAuth token...
2026-05-11 11:08:56,100 [INFO] Token refreshed; valid until 12:07:15
2026-05-11 11:09:12,694 [ERROR]   [FAIL] 21186140 — E-mail Id is mandatory.; E-mail Id is mandatory.
2026-05-11 11:09:13,031 [ERROR]   [FAIL] 25928300 — Invalid contact id.
2026-05-11 11:09:13,037 [ERROR]   [FAIL] 25262934 — Invalid contact id.
2026-05-11 11:09:13,056 [ERROR]   [FAIL] 12579173 — No Such Account Exists.
2026-05-11 11:09:13,057 [ERROR]   [FAIL] 25342741 — No Such Account Exists.
2026-05-11 11:09:13,057 [ERROR]   [FAIL] 24000898 — No Such Account Exists.
2026-05-11 11:09:13,654 [ERROR]   [FAIL] 16673533 — Invalid contact id.
2026-05-11 11:09:13,665 [ERROR]   [FAIL] 26302594 — No Such Account Exists.
2026-05-11 11:09:13,670 [ERROR]   [FAIL] 27386946 — Invalid contact id.
2026-05-11 11:09:13,678 [ERROR]   [FAIL] 10625002 — No Such Account Exists.
2026-05-11 11:09:13,6


=== Profiling Summary ===
Total wall time:          19.8s
Throughput:               0.5 contacts/s
Avg build time per row:   2.2ms
Avg network time per row: 19302.6ms
Max network time:         19756.4ms
P95 network time:         19734.3ms

Failed rows (10):


,account_code,contact_code,first_name,last_name,status,error,elapsed_build_ms,elapsed_net_ms
0,21186140,BILLING-21186140,Lyn-Marie,Harris,failed,E-mail Id is mandatory.; E-mail Id is mandatory.,0.9,18741.2
1,25928300,BILLING-25928300,Jon,Verhoek,failed,Invalid contact id.,0.8,19060.4
2,25262934,BILLING-25262934,Rata,Miller,failed,Invalid contact id.,4.5,19071.0
3,12579173,BILLING-12579173,Cara Tipping Smith t/as Copy Carats,,failed,No Such Account Exists.,1.3,19115.8
4,25342741,BILLING-25342741,Cliff,Black,failed,No Such Account Exists.,0.9,19088.2
5,24000898,BILLING-24000898,Kylie,Glenn,failed,No Such Account Exists.,1.0,19099.8
6,16673533,BILLING-16673533,Jason,Yee,failed,Invalid contact id.,1.7,19707.3
7,26302594,BILLING-26302594,Nathan,Kennedy,failed,No Such Account Exists.,1.2,19692.1
8,27386946,BILLING-27386946,Dana,Waitai-Cross,failed,Invalid contact id.,0.9,19694.2
9,10625002,BILLING-10625002,Anna,Milroy,failed,No Such Account Exists.,9.0,19756.4


In [11]:
# Export failures to CSV for manual review / retry
failures.to_csv('Failed_Contact_Migrations.csv', index=False)
print(f'Exported {len(failures):,} failed rows to Failed_Contact_Migrations.csv')

Exported 10 failed rows to Failed_Contact_Migrations.csv
